In [1]:
import os
import pandas as pd
from datetime import datetime
import importlib
import numpy as np

np.set_printoptions(suppress=True)

In [2]:
# Relative imports
os.chdir("..")
import hidden_state_model.processor
importlib.reload(hidden_state_model.processor)
Processor = hidden_state_model.processor.Processor
os.chdir("hidden_state_model")

### Read (and compact) dataframes

In [3]:
compact = True

In [4]:
# Iterate over files in dfs/*.parquet and combine to one df
dfs = []

read = []
for file in os.listdir("data"):
    if file.endswith(".parquet"):
        read.append(file)
        print(f"Reading {file}")
        df = pd.read_parquet(f"data/{file}")
        dfs.append(df)
    if file.endswith(".csv"):
        read.append(file)
        df = pd.read_csv(f"data/{file}", index_col=0)
        dfs.append(df)

raw_df = pd.concat(dfs)

if compact and len(dfs) > 10:
    print("Compacintg dfs")
    # Move read files to trash and write combined df to dfs/combined_{timestamp}.parquet
    trash = "data/trash"
    for f in read:
        os.rename(f"data/{f}", f"{trash}/{f}")

    timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
    raw_df.to_parquet(f"data/compacted_{timestamp}.parquet")

dfs = []  # Clear memory
raw_df

Reading fixed.parquet


,prev_entry,public_cards,player_piles,current_player_i,bet_in_stage,bet_in_game,player_has_played,player_is_folded,first_better_i,big_blind,player_name,player_type,action,amount,p,relative_ev,rank,tiebreakers
state_id,,,,,,,,,,,,,,,,,,
4896996752,<NA>,[],"[98, 96]",0,"[2, 4]","[2, 4]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,call,2,0.5130,0.015390,0,"[8, 4, 0, 0, 0]"
6158719728,4896996752,"[41, 30, 42]","[96, 96]",0,"[0, 0]","[4, 4]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,raise,4,0.6077,0.024308,1,"[4, 8, 3, 2, 0]"
6158687296,6158719728,"[41, 30, 42, 6]","[92, 92]",0,"[0, 0]","[8, 8]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,raise,8,0.5770,0.046160,1,"[4, 8, 6, 3, 0]"
6158493008,6158687296,"[41, 30, 42, 6, 28]","[84, 84]",0,"[0, 0]","[16, 16]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,check,0,0.5421,0.086736,2,"[4, 2, 8, 0, 0]"
6158678432,6158493008,"[41, 30, 42, 6, 28]","[84, 78]",0,"[0, 6]","[16, 22]","[True, True]","[False, False]",0,4,Tord,HumanPlayer,call,6,0.5421,0.102999,2,"[4, 2, 8, 0, 0]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6431248928,6127529072,"[8, 9, 36]","[152, 40]",0,"[0, 0]","[4, 4]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,raise,4,0.6240,0.024960,1,"[8, 10, 9, 4, 0]"
6431104400,6431248928,"[8, 9, 36, 6]","[148, 36]",0,"[0, 0]","[8, 8]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,raise,4,0.5459,0.043672,1,"[8, 10, 9, 6, 0]"
6127537840,6431104400,"[8, 9, 36, 6]","[144, 26]",0,"[4, 10]","[12, 18]","[True, True]","[False, False]",0,4,Tord,HumanPlayer,call,6,0.5459,0.081885,1,"[8, 10, 9, 6, 0]"


In [5]:
raw_df.dtypes

prev_entry            object
public_cards          object
player_piles          object
current_player_i       int64
bet_in_stage          object
bet_in_game           object
player_has_played     object
player_is_folded      object
first_better_i         int64
big_blind              int64
player_name           object
player_type           object
action                object
amount                 int64
p                    float64
relative_ev          float64
rank                   int64
tiebreakers           object
dtype: object

In [6]:
# Check for conflicting rows
dupe_df = raw_df[raw_df.index.duplicated()]
assert len(dupe_df) == 0, dupe_df

## Process data

In [7]:
processor = Processor(raw_df)
df = processor.get_processed_df()
df

,game_id,raise_preflop,raise_flop,raise_turn,raise_river,raise_showdown,call_preflop,call_flop,call_turn,call_river,...,opponent_check_turn,opponent_check_river,opponent_check_showdown,action,amount,excess_rank,p,relative_ev,stage,player_name
4896996752,c6467de0-1f25-4140-b37d-e6f9bd23eafb,0,0,0,0,0,0,0,0,0,...,0,0,0,call,2,0,0.5130,0.015390,preflop,Tord
6158719728,c6467de0-1f25-4140-b37d-e6f9bd23eafb,0,0,0,0,0,2,0,0,0,...,0,0,0,raise,4,1,0.6077,0.024308,flop,Tord
6158687296,c6467de0-1f25-4140-b37d-e6f9bd23eafb,0,4,0,0,0,2,0,0,0,...,0,0,0,raise,8,1,0.5770,0.046160,turn,Tord
6158493008,c6467de0-1f25-4140-b37d-e6f9bd23eafb,0,4,8,0,0,2,0,0,0,...,0,0,0,check,0,1,0.5421,0.086736,river,Tord
6158678432,c6467de0-1f25-4140-b37d-e6f9bd23eafb,0,4,8,0,0,2,0,0,0,...,0,0,0,call,6,1,0.5421,0.102999,river,Tord
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6431248928,7aaa5f37-9c07-4b24-9100-c9ebd3d1909d,0,0,0,0,0,2,0,0,0,...,0,0,0,raise,4,1,0.6240,0.024960,flop,Tord
6431104400,7aaa5f37-9c07-4b24-9100-c9ebd3d1909d,0,4,0,0,0,2,0,0,0,...,0,0,0,raise,4,1,0.5459,0.043672,turn,Tord
6127537840,7aaa5f37-9c07-4b24-9100-c9ebd3d1909d,0,4,4,0,0,2,0,0,0,...,0,0,0,call,6,1,0.5459,0.081885,turn,Tord
6431250896,7aaa5f37-9c07-4b24-9100-c9ebd3d1909d,0,4,4,0,0,2,0,6,0,...,0,1,0,check,0,0,0.9412,0.169416,river,Tord


In [8]:
df.dtypes

game_id                     object
raise_preflop                int64
raise_flop                   int64
raise_turn                   int64
raise_river                  int64
raise_showdown               int64
call_preflop                 int64
call_flop                    int64
call_turn                    int64
call_river                   int64
call_showdown                int64
check_preflop                int64
check_flop                   int64
check_turn                   int64
check_river                  int64
check_showdown               int64
opponent_raise_preflop       int64
opponent_raise_flop          int64
opponent_raise_turn          int64
opponent_raise_river         int64
opponent_raise_showdown      int64
opponent_call_preflop        int64
opponent_call_flop           int64
opponent_call_turn           int64
opponent_call_river          int64
opponent_call_showdown       int64
opponent_check_preflop       int64
opponent_check_flop          int64
opponent_check_turn 

## Training

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [10]:
X = df.drop(["excess_rank", "game_id", "p", "relative_ev"], axis=1)
y = df["excess_rank"]
groups = df["game_id"]  # Group by 'game_id' to ensure no data leakage

In [11]:
X

,raise_preflop,raise_flop,raise_turn,raise_river,raise_showdown,call_preflop,call_flop,call_turn,call_river,call_showdown,...,opponent_call_showdown,opponent_check_preflop,opponent_check_flop,opponent_check_turn,opponent_check_river,opponent_check_showdown,action,amount,stage,player_name
4896996752,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,call,2,preflop,Tord
6158719728,0,0,0,0,0,2,0,0,0,0,...,0,0,1,0,0,0,raise,4,flop,Tord
6158687296,0,4,0,0,0,2,0,0,0,0,...,0,0,1,0,0,0,raise,8,turn,Tord
6158493008,0,4,8,0,0,2,0,0,0,0,...,0,0,1,0,0,0,check,0,river,Tord
6158678432,0,4,8,0,0,2,0,0,0,0,...,0,0,1,0,0,0,call,6,river,Tord
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6431248928,0,0,0,0,0,2,0,0,0,0,...,0,0,1,0,0,0,raise,4,flop,Tord
6431104400,0,4,0,0,0,2,0,0,0,0,...,0,0,1,0,0,0,raise,4,turn,Tord
6127537840,0,4,4,0,0,2,0,0,0,0,...,0,0,1,0,0,0,call,6,turn,Tord
6431250896,0,4,4,0,0,2,0,6,0,0,...,0,0,1,0,1,0,check,0,river,Tord


In [12]:
y

4896996752    0
6158719728    1
6158687296    1
6158493008    1
6158678432    1
             ..
6431248928    1
6431104400    1
6127537840    1
6431250896    0
6431104448    0
Name: excess_rank, Length: 3741, dtype: int64

In [13]:
# Identify categorical columns (excluding 'game_id')
categorical_cols = ["action", "stage", "player_name"]

# Preprocessing pipeline: OneHotEncoding for categorical and scaling for numerical
preprocessor = ColumnTransformer(
    transformers=[("cat", OneHotEncoder(drop="first"), categorical_cols)],
    remainder="passthrough",
)

# Create the full pipeline with logistic regression
model = Pipeline(
    [
        ("preprocess", preprocessor),
        (
            "classifier",
            LogisticRegression(
                multi_class="multinomial", solver="lbfgs", max_iter=1000
            ),
        ),
    ]
)

In [14]:
# Grouped train-test split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")

Train shape: (2989, 34)
Test shape: (752, 34)


In [15]:
# Train the model
model.fit(X_train, y_train)

# Evaluate the model
accuracy = model.score(X_test, y_test)
print(f"Accuracy: {accuracy}")

Accuracy: 0.726063829787234


In [16]:
probabilities = model.predict_proba(X_test)
prob_df = pd.DataFrame(probabilities, columns=model.classes_, index=X_test.index)
prob_df["true"] = y_test.values
prob_df["pred"] = model.predict(X_test)
prob_df["correct"] = prob_df["true"] == prob_df["pred"]
prob_df["goodness"] = prob_df.apply(lambda x: x.get(x["true"]) or 0, axis=1)
print("Accuracy", prob_df["correct"].mean())
print("Mean goodness: ", prob_df["goodness"].mean())
prob_df

Accuracy 0.726063829787234
Mean goodness:  0.6711258072918275


,0,1,2,3,4,5,true,pred,correct,goodness
4896996752,0.961683,0.037610,5.187215e-05,0.000170,0.000267,0.000218,0,0,True,0.961683
6158719728,0.108848,0.763283,8.703751e-02,0.010322,0.015430,0.015080,1,1,True,0.763283
6158687296,0.049170,0.741412,1.559945e-01,0.009223,0.019964,0.024236,1,1,True,0.741412
6158493008,0.325359,0.580307,1.577383e-02,0.009809,0.022932,0.045820,1,1,True,0.580307
6158678432,0.385470,0.574513,5.255242e-03,0.000219,0.013654,0.020888,1,1,True,0.574513
...,...,...,...,...,...,...,...,...,...,...
6127527776,0.963925,0.032042,5.317546e-04,0.001296,0.001023,0.001181,0,0,True,0.963925
6431193840,0.722296,0.262021,6.700131e-03,0.003120,0.003244,0.002618,0,0,True,0.722296
6127534384,0.995860,0.004094,2.757217e-07,0.000033,0.000002,0.000011,0,0,True,0.995860
6431194992,0.722296,0.262021,6.700131e-03,0.003120,0.003244,0.002618,1,0,False,0.262021


In [17]:
# Look at incorrect predictions
print(prob_df[prob_df["correct"] == False].shape[0], "incorrect predictions:")
prob_df[prob_df["correct"] == False]

206 incorrect predictions:


,0,1,2,3,4,5,true,pred,correct,goodness
6064258592,0.929362,0.069534,0.000286,0.000385,0.000221,0.000212,1,0,False,0.069534
49cf603d-3b9b-42d7-a2fc-57cf53b5fcb6,0.910595,0.088489,0.000353,0.000202,0.000239,0.000123,1,0,False,0.088489
6064328176,0.637611,0.328290,0.021854,0.002157,0.007411,0.002678,1,0,False,0.328290
6064256288,0.053118,0.507215,0.369157,0.013904,0.045179,0.011427,3,1,False,0.013904
6064224576,0.043262,0.387638,0.488232,0.013471,0.051562,0.015835,3,2,False,0.013471
...,...,...,...,...,...,...,...,...,...,...
6101517360,0.322664,0.496779,0.030626,0.072310,0.025519,0.052103,0,1,False,0.322664
6127312464,0.644763,0.287630,0.006141,0.017736,0.021210,0.022520,5,0,False,0.022520
6127438960,0.600558,0.257153,0.002981,0.041777,0.045092,0.052440,5,0,False,0.052440
6431194992,0.722296,0.262021,0.006700,0.003120,0.003244,0.002618,1,0,False,0.262021


### Attempt to infer card probabilities from rank and table

In [45]:
from cpp_poker.cpp_poker import Hand, CardCollection, HandRank

In [48]:
hand_ranks_per_state = []
table_ranks = []
for i, (state_id, row) in enumerate(prob_df.iterrows()):
    raw_row = raw_df.loc[state_id]
    table_cards = CardCollection(raw_row["public_cards"])
    table_rank = table_cards.rank_hand().get_rank()
    excess_hand_ranks_row = table_cards.rank_all_hands()
    hand_ranks_per_state.append(
        [rank.get_rank() - table_rank for rank in excess_hand_ranks_row]
    )
    table_ranks.append(table_rank)

table_ranks_df = pd.DataFrame(table_ranks, index=prob_df.index, columns=["table_rank"])
hand_ranks_df = pd.DataFrame(hand_ranks_per_state, index=prob_df.index)
hand_ranks_df

,0,1,2,3,4,5,6,7,8,9,...,1316,1317,1318,1319,1320,1321,1322,1323,1324,1325
4896996752,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6158719728,4,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6158687296,4,1,1,1,4,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6158493008,3,2,1,1,3,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6158678432,3,2,1,1,3,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6127527776,0,0,0,0,0,1,0,0,0,0,...,4,0,0,0,0,0,0,0,0,0
6431193840,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6127534384,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6431194992,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [31]:
# Get the values of the column indices from hand_ranks_df
column_indices = hand_ranks_df.values

# Use np.arange to create an array of row indices
row_indices = np.arange(hand_ranks_df.shape[0])[:, None]

# Use the row and column indices to index into the DataFrame values
hand_probs_df = pd.DataFrame(
    prob_df.values[row_indices, column_indices],
    index=hand_ranks_df.index,
    columns=hand_ranks_df.columns,
)

# Normalize the hand probabilities to sum to 1 for each row
hand_probs_df = hand_probs_df.div(hand_probs_df.sum(axis=1), axis=0)
hand_probs_df

,0,1,2,3,4,5,6,7,8,9,...,1316,1317,1318,1319,1320,1321,1322,1323,1324,1325
4896996752,0.000799,0.000799,0.000799,0.000799,0.000799,0.000799,0.000799,0.000799,0.000799,0.000799,...,0.000799,0.000799,0.000799,0.000799,0.000799,0.000799,0.000799,0.000799,0.000799,0.000799
6158719728,0.000036,0.001771,0.001771,0.001771,0.000253,0.000253,0.000253,0.000253,0.000253,0.000253,...,0.000253,0.000253,0.000253,0.000253,0.000253,0.000253,0.000253,0.000253,0.000253,0.000253
6158687296,0.00005,0.00187,0.00187,0.00187,0.00005,0.000124,0.000124,0.000124,0.000124,0.000124,...,0.000124,0.000124,0.000124,0.000124,0.000124,0.000124,0.000124,0.000124,0.000124,0.000124
6158493008,0.000022,0.000036,0.001324,0.001324,0.000022,0.000743,0.000743,0.000743,0.000743,0.000743,...,0.000743,0.000743,0.000743,0.000743,0.000743,0.000743,0.000743,0.000743,0.000743,0.000743
6158678432,0.0,0.000011,0.001226,0.001226,0.0,0.000823,0.000823,0.000823,0.000823,0.000823,...,0.000823,0.000823,0.000823,0.000823,0.000823,0.000823,0.000823,0.000823,0.000823,0.000823
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6127527776,0.001235,0.001235,0.001235,0.001235,0.001235,0.000041,0.001235,0.001235,0.001235,0.001235,...,0.000001,0.001235,0.001235,0.001235,0.001235,0.001235,0.001235,0.001235,0.001235,0.001235
6431193840,0.000784,0.000784,0.000784,0.000784,0.000784,0.000784,0.000784,0.000784,0.000784,0.000784,...,0.000784,0.000784,0.000784,0.000784,0.000784,0.000784,0.000784,0.000784,0.000784,0.000784
6127534384,0.000801,0.000801,0.000801,0.000801,0.000801,0.000801,0.000801,0.000801,0.000801,0.000801,...,0.000801,0.000801,0.000801,0.000801,0.000801,0.000801,0.000801,0.000801,0.000801,0.000801
6431194992,0.000784,0.000784,0.000784,0.000784,0.000784,0.000784,0.000784,0.000784,0.000784,0.000784,...,0.000784,0.000784,0.000784,0.000784,0.000784,0.000784,0.000784,0.000784,0.000784,0.000784


In [56]:
# Look at the max probabilites for different hands
mask = prob_df["pred"] == 2
max_probs = hand_probs_df[mask].max(axis=1)
max_prob_hands = hand_probs_df[mask].idxmax(axis=1)
max_prob_df = pd.DataFrame({"max_prob": max_probs, "hand": max_prob_hands})
max_prob_df["hand"] = max_prob_df["hand"].apply(lambda x: Hand(x).str())
max_prob_df["table"] = raw_df.loc[max_prob_df.index, "public_cards"].apply(
    lambda x: CardCollection(x).str()
)
max_prob_df["predicted_excess_rank"] = prob_df.loc[max_prob_df.index, "pred"]
max_prob_df["pred_rank"] = max_prob_df["predicted_excess_rank"] + table_ranks_df.loc[
    max_prob_df.index, "table_rank"
]
max_prob_df["pred_rank_str"] = max_prob_df["pred_rank"].apply(
    lambda x: HandRank(x, []).get_rank_name()
)
max_prob_df = max_prob_df.drop(["predicted_excess_rank"], axis=1)
max_prob_df

,max_prob,hand,table,pred_rank,pred_rank_str
6064224576,0.001002,♥ 2 ♠ 7,♥ 7 ♦ 7 ♣ 7 ♠ A,5,Flush
6064257536,0.003634,♥ 2 ♠ 7,♥ 7 ♦ 7 ♣ 7 ♠ A,5,Flush
6064223136,0.003058,♥ 2 ♠ 7,♥ 7 ♦ 7 ♦ J ♣ 7 ♠ A,5,Flush
6082204672,0.002584,♥ 5 ♥ 6,♥ A ♦ 6 ♣ Q ♠ 5,2,Two Pairs
6085936656,0.006707,♥ 5 ♥ 6,♥ A ♦ 6 ♣ Q ♠ 5,2,Two Pairs
6086080400,0.008954,♥ 5 ♥ 6,♥ A ♦ 6 ♣ Q ♠ 5,2,Two Pairs
6086591584,0.004264,♥ 5 ♥ 6,♥ A ♦ 6 ♣ Q ♠ 5 ♠ J,2,Two Pairs
5219707776,0.002233,♥ 6 ♥ 7,♥ 10 ♥ A ♦ 7 ♣ 6,2,Two Pairs
6103977408,0.00484,♥ 6 ♥ 7,♥ 10 ♥ A ♦ 7 ♣ 6,2,Two Pairs
6103977600,0.008858,♥ 6 ♥ 7,♥ 10 ♥ A ♦ 7 ♣ 6,2,Two Pairs
